# Foursight: m6A prediction

This notebook runs our m6A prediction script from a fresh copy of the
[repository](https://github.com/AsJayTee/DSA4262-Foursight-Inference) on a 1,000-site
test dataset that ships with the code.

**To run it, choose Runtime → Run all.** It takes about a minute. It needs
no GPU and no credentials, and it does not use the training script.

Steps 1-5 cover the required evaluation. Step 6 is optional and predicts on
your own file or on a full SG-NEx sample.

## 1. Get the code

In [ ]:
![ -d /content/DSA4262-Foursight-Inference ] || git clone --depth 1 https://github.com/AsJayTee/DSA4262-Foursight-Inference.git /content/DSA4262-Foursight-Inference
%cd /content/DSA4262-Foursight-Inference

## 2. Install

Installs the package and its three dependencies (numpy, pandas and
scikit-learn). The shipped model runs in plain numpy, so torch is not needed.

In [ ]:
!pip install -q .

## 3. Predict on the test dataset

`data/sample/sample.json.gz` is in the course's input format (the m6Anet-processed
`data.json`), gzipped.

In [ ]:
!python scripts/predict.py --model models/final --input data/sample/sample.json.gz --output predictions.csv

## 4. Check the output format

Checks that the columns are exactly `transcript_id,transcript_position,score`,
that every score is a number in [0, 1], and that the CSV has exactly one row for
every site in the input. Prints `OK` or lists each problem.

In [ ]:
!python -c "from m6a.evaluation import validate_submission; problems = validate_submission('predictions.csv', 'data/sample/sample.json.gz'); print(chr(10).join(problems) or 'OK: predictions.csv has the required format and one row per input site')"

## 5. Look at the predictions

A higher score means the site is more likely to carry m6A. The score is each
site's rank within the file, averaged over the six networks of the ensemble, so
it orders sites but is not a calibrated probability.

In [ ]:
import pandas as pd

predictions = pd.read_csv("predictions.csv")
print(f"{len(predictions):,} sites scored")
predictions.head(10)

In [ ]:
# Download the CSV to your computer (optional)
from google.colab import files
files.download("predictions.csv")

## 6. Optional: predict on other data

Either of the cells below works. Both reuse the commands from steps 3 and 4.
Run All stops at the upload box in 6a: choose a file, or skip it and run 6b on its own.

**6a. Your own file:** upload any `data.json` or `data.json.gz` in the
m6Anet-processed format.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_file = next(iter(uploaded))
!python scripts/predict.py --model models/final --input "{input_file}" --output my_predictions.csv
!python -c "from m6a.evaluation import validate_submission; problems = validate_submission('my_predictions.csv', '{input_file}'); print(chr(10).join(problems) or 'OK: my_predictions.csv has the required format and one row per input site')"

**6b. A full SG-NEx sample:** one direct RNA-seq run from the public
[SG-NEx bucket](https://registry.opendata.aws/sgnex/). It needs no credentials.
The default, an HCT116 run, is 1.5 GB (about 1.3 million sites), so it takes
several minutes: about 5 on an 8-core laptop, longer on Colab's smaller machine.
Prediction peaks at 5.6 GB of memory, which fits in Colab's free tier.

To try another cell line, change `SAMPLE` to one of the runs listed in the
cell. Memory grows with file size, so runs above about 2 GB may not fit in
Colab's free tier.

In [ ]:
SAMPLE = "SGNex_Hct116_directRNA_replicate3_run4"
# Other runs, with their file size:
#   SGNex_A549_directRNA_replicate5_run1   0.8 GB    SGNex_A549_directRNA_replicate6_run1   1.9 GB
#   SGNex_H9_directRNA_replicate2_run2     0.6 GB    SGNex_H9_directRNA_replicate4_run2     0.5 GB
#   SGNex_H9_directRNA_replicate3_run1     1.1 GB    SGNex_H9_directRNA_replicate4_run1     1.7 GB
#   SGNex_HepG2_directRNA_replicate6_run1  2.4 GB    SGNex_HepG2_directRNA_replicate5_run2  2.6 GB
#   SGNex_K562_directRNA_replicate4_run1   SGNex_MCF7_directRNA_replicate4_run1   SGNex_HEYA8_directRNA_replicate1_run1
!curl -s -o /content/sgnex.json https://sg-nex-data.s3.amazonaws.com/data/processed_data/m6Anet/{SAMPLE}/data.json
!python scripts/predict.py --model models/final --input /content/sgnex.json --output sgnex_predictions.csv
!python -c "from m6a.evaluation import validate_submission; problems = validate_submission('sgnex_predictions.csv', '/content/sgnex.json'); print(chr(10).join(problems) or 'OK: sgnex_predictions.csv has the required format and one row per input site')"